# Generalized MCLP Pipeline — Any City with Demand Points

A standalone, reusable version of the base optimization pipeline (no constraint filtering, no equity weighting) that runs on **any city** given a file of demand points with latitude/longitude columns.

**Minimum input:** one file (CSV, Parquet, GeoPackage, …) with a lat column and a lon column.  
The street network is fetched automatically from OpenStreetMap via OSMnx.  
The projected CRS (meters) is detected automatically from the data's median coordinates.

**Test city:** Porto, Portugal — Porto taxi trips, June 2014 (matching the single-month approach used for NYC).

---

## Scope
| In scope | Out of scope |
|---|---|
| NKDE demand scoring | NYC-specific constraint filtering |
| Candidate generation | Equity / demographic weighting |
| MCLP optimization | Any city-specific layers |
| Coverage curve + maps | |

---

## Part A — Download and parse the Porto dataset

**Download the Porto taxi dataset before running Part A.**

1. Create a free Kaggle account and accept the competition rules at  
   `https://www.kaggle.com/c/pkdd-15-predict-taxi-service-trajectory-i`
2. Download **train.csv.zip**, extract it, place `train.csv` in `data/raw/`
3. Run the cell below — it filters to **June 2014** and saves `data/raw/porto_pickups.parquet`
4. Then proceed to Part B

The dataset spans 2013-07-01 → 2014-06-30. June 2014 is the final (and most complete) month.

> **Note:** pandana requires the `network.py` int32 patch on Windows (already applied in this environment during notebook 04).

In [ ]:
# ── Part A: Parse Porto taxi CSV — June 2014 only ──────────────────────────
# The Porto dataset spans 2013-07-01 → 2014-06-30 (one full year).
# We filter to June 2014 to match the single-month approach used for NYC.
#
# Kaggle  → TIMESTAMP column (Unix seconds UTC); filter 2014-06-01..2014-06-30
# figshare→ checks for a date column; filters same window if present
#
# Input:  data/raw/train.csv        (Kaggle)
#      or data/raw/porto_taxi.csv   (figshare)
# Output: data/raw/porto_pickups.parquet

import ast
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR  = PROJECT_ROOT / "data/raw"
OUT_PARQ = RAW_DIR / "porto_pickups.parquet"

# June 2014 window in UTC Unix seconds
JUNE_START = int(pd.Timestamp("2014-06-01", tz="UTC").timestamp())  # 1401580800
JUNE_END   = int(pd.Timestamp("2014-07-01", tz="UTC").timestamp())  # 1404172800

_CANDIDATES = ["train.csv", "porto_taxi.csv"]
RAW_CSV = next((RAW_DIR / f for f in _CANDIDATES if (RAW_DIR / f).exists()), None)

if RAW_CSV is None:
    raise FileNotFoundError(
        "\n\nPorto taxi CSV not found in data/raw/.\n"
        "Download train.csv from Kaggle:\n"
        "  https://www.kaggle.com/c/pkdd-15-predict-taxi-service-trajectory-i\n"
        "Extract train.csv.zip → place train.csv in data/raw/ → re-run this cell.\n"
    )

print(f"Reading {RAW_CSV.name} …")
raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"Rows before filtering: {len(raw):,}")
print(f"Columns: {list(raw.columns[:12])} …")

_WKT_RE = re.compile(r"POINT\(([\d.\-]+)\s+([\d.\-]+)\)", re.IGNORECASE)

if "POLYLINE" in raw.columns:
    # ── Kaggle format ───────────────────────────────────────────────────────
    print("Detected Kaggle format (POLYLINE + TIMESTAMP).")

    if "MISSING_DATA" in raw.columns:
        n_before = len(raw)
        raw = raw[raw["MISSING_DATA"] != True].copy()
        print(f"  Dropped {n_before - len(raw):,} MISSING_DATA rows")

    raw = raw[raw["POLYLINE"].fillna("").str.strip() != "[]"].copy()

    if "TIMESTAMP" in raw.columns:
        n_before = len(raw)
        raw = raw[(raw["TIMESTAMP"] >= JUNE_START) & (raw["TIMESTAMP"] < JUNE_END)].copy()
        print(f"  June 2014 filter: {n_before:,} → {len(raw):,} rows")
    else:
        print("  WARNING: no TIMESTAMP column — using all rows (no month filter)")

    def _parse_polyline(val):
        try:
            coords = ast.literal_eval(str(val))
            if coords and len(coords[0]) == 2:
                return float(coords[0][0]), float(coords[0][1])  # lon, lat
        except Exception:
            pass
        return None, None

    parsed = raw["POLYLINE"].apply(_parse_polyline)
    raw["pickup_lon"] = parsed.apply(lambda t: t[0])
    raw["pickup_lat"] = parsed.apply(lambda t: t[1])

elif "source_point" in raw.columns:
    # ── figshare format ─────────────────────────────────────────────────────
    print("Detected figshare format (source_point / WKT).")

    date_col = next((c for c in raw.columns if c.lower() in ("timestamp", "datetime", "date", "time")), None)
    if date_col:
        try:
            ts = pd.to_datetime(raw[date_col], utc=True, errors="coerce").astype("int64") // 10**9
            n_before = len(raw)
            raw = raw[(ts >= JUNE_START) & (ts < JUNE_END)].copy()
            print(f"  June 2014 filter on '{date_col}': {n_before:,} → {len(raw):,} rows")
        except Exception as e:
            print(f"  WARNING: could not parse '{date_col}' ({e}) — using all rows")
    else:
        print("  WARNING: no date column found — using all rows (no month filter)")

    def _parse_wkt(val):
        if pd.isna(val):
            return None, None
        m = _WKT_RE.search(str(val))
        if not m:
            return None, None
        return float(m.group(1)), float(m.group(2))

    parsed = raw["source_point"].apply(_parse_wkt)
    raw["pickup_lon"] = parsed.apply(lambda t: t[0])
    raw["pickup_lat"] = parsed.apply(lambda t: t[1])

else:
    raise ValueError(
        f"Unrecognised format in {RAW_CSV.name}.\n"
        "Expected 'POLYLINE' (Kaggle) or 'source_point' (figshare).\n"
        f"Columns found: {list(raw.columns)}"
    )

# ── Clean ──────────────────────────────────────────────────────────────────
pickups = raw[["pickup_lon", "pickup_lat"]].copy().dropna()
pickups = pickups[(pickups["pickup_lon"] != 0) & (pickups["pickup_lat"] != 0)]
pickups = pickups.reset_index(drop=True)

print(f"Rows after cleaning:  {len(pickups):,}")
print(pickups.describe())

OUT_PARQ.parent.mkdir(parents=True, exist_ok=True)
pickups.to_parquet(OUT_PARQ, index=False)
print(f"\nSaved → {OUT_PARQ.resolve()}")

---
## Part B — Config-driven generalized pipeline

In [1]:
CONFIG = {
    "demand_path": "data/raw/porto_pickups.parquet",
    "lat_col": "pickup_lat",
    "lon_col": "pickup_lon",
    "place_name": "Porto, Portugal",
    "bbox": None,           # optional (north, south, east, west); overrides place_name if set
    "coverage_radius_m": 500,
    "candidate_spacing_m": 800,
    "nkde_bandwidth_m": 500,
    "p_values": [20, 40, 55, 70, 85, 100],
    "excluded_highways": ["motorway", "motorway_link", "trunk", "trunk_link", "raceway", "service"],
    "output_prefix": "porto",
}

In [2]:
# ── Imports ────────────────────────────────────────────────────────────────
import ast
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import pandana
import pulp
from scipy.spatial import cKDTree
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [3]:
def detect_utm_crs(lon: float, lat: float) -> str:
    """Return EPSG code for the UTM zone containing (lon, lat)."""
    zone = int((lon + 180) / 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"


def load_demand(config: dict):
    """Load demand points, detect UTM CRS from median lon/lat, reproject."""
    path = PROJECT_ROOT / config["demand_path"]
    ext = path.suffix.lower()

    if ext == ".parquet":
        df = pd.read_parquet(path)
    elif ext == ".csv":
        df = pd.read_csv(path)
    else:
        df = gpd.read_file(path)

    lat_col = config["lat_col"]
    lon_col = config["lon_col"]

    df = df[[lon_col, lat_col]].dropna()
    df = df[(df[lon_col] != 0) & (df[lat_col] != 0)]
    print(f"Demand rows loaded: {len(df):,}")

    med_lon = float(df[lon_col].median())
    med_lat = float(df[lat_col].median())
    target_crs = detect_utm_crs(med_lon, med_lat)
    print(f"Detected UTM CRS: {target_crs}  (median lon={med_lon:.4f}, lat={med_lat:.4f})")

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
    ).to_crs(target_crs)

    return gdf, target_crs


def pull_network(config: dict, target_crs: str):
    """Download drivable OSM network for the city and reproject to target_crs."""
    bbox = config.get("bbox")
    if bbox:
        north, south, east, west = bbox
        G = ox.graph_from_bbox(north, south, east, west, network_type="drive")
    else:
        G = ox.graph_from_place(config["place_name"], network_type="drive")

    G = ox.project_graph(G, to_crs=target_crs)
    nodes, edges = ox.graph_to_gdfs(G)
    nodes = nodes.to_crs(target_crs)
    edges = edges.to_crs(target_crs)
    edges["length_m"] = edges.geometry.length

    # graph_to_gdfs keeps u/v/key in the MultiIndex; bring them into columns
    if "u" not in edges.columns or "v" not in edges.columns:
        edges = edges.reset_index(drop=False)
    # osmid is the node index; bring it into a column for mapping
    if "osmid" not in nodes.columns:
        nodes = nodes.reset_index(drop=False)

    print(f"Network: {len(nodes):,} nodes, {len(edges):,} edges")
    return G, nodes, edges


def compute_nkde(edges: gpd.GeoDataFrame, demand: gpd.GeoDataFrame, config: dict, G) -> gpd.GeoDataFrame:
    """
    Snap demand to nearest edges, aggregate demand_count, apply quadratic
    kernel NKDE at nkde_bandwidth_m, log1p-normalize to nkde_score 0-1.
    Same logic as 02_nkde.ipynb.
    """
    from collections import defaultdict

    bandwidth = config["nkde_bandwidth_m"]
    chunk = 50_000

    # ── Snap demand to edges ────────────────────────────────────────────────
    n = len(demand)
    edge_u = np.empty(n, dtype=np.int64)
    edge_v = np.empty(n, dtype=np.int64)

    print(f"Snapping {n:,} demand points to edges …")
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        sub = demand.iloc[start:end]
        uvk = ox.nearest_edges(G, sub.geometry.x.to_numpy(), sub.geometry.y.to_numpy())
        edge_u[start:end] = np.fromiter((t[0] for t in uvk), dtype=np.int64, count=end - start)
        edge_v[start:end] = np.fromiter((t[1] for t in uvk), dtype=np.int64, count=end - start)
        print(f"  {end:,} / {n:,} ({100*end/n:.0f}%)")

    dc_df = pd.DataFrame({"edge_u": edge_u, "edge_v": edge_v})
    agg = dc_df.groupby(["edge_u", "edge_v"]).size().reset_index(name="demand_count")

    edges_out = edges.copy()
    edges_out = edges_out.merge(agg, left_on=["u", "v"], right_on=["edge_u", "edge_v"], how="left")
    edges_out = edges_out.drop(columns=["edge_u", "edge_v"], errors="ignore")
    edges_out["demand_count"] = edges_out["demand_count"].fillna(0).astype(np.int64)

    # ── Quadratic NKDE ──────────────────────────────────────────────────────
    eu = edges_out["u"].to_numpy()
    ev = edges_out["v"].to_numpy()
    dc_arr = edges_out["demand_count"].to_numpy(dtype=np.float64)
    n_edges = len(edges_out)

    midpoints = edges_out.geometry.interpolate(0.5, normalized=True)
    nearest_nodes = np.asarray(
        ox.distance.nearest_nodes(G, midpoints.x.to_numpy(), midpoints.y.to_numpy())
    )

    node_incident: dict = defaultdict(set)
    for j in range(n_edges):
        node_incident[int(eu[j])].add(j)
        node_incident[int(ev[j])].add(j)

    raw_scores = np.zeros(n_edges, dtype=np.float64)
    length_cache: dict = {}
    t0 = time.perf_counter()

    print(f"Computing NKDE over {n_edges:,} edges (bandwidth={bandwidth} m) …")
    for i in range(n_edges):
        origin = int(nearest_nodes[i])
        if origin not in length_cache:
            length_cache[origin] = nx.single_source_dijkstra_path_length(
                G, origin, cutoff=bandwidth, weight="length"
            )
        lengths = length_cache[origin]
        cands_j: set = set()
        for nd in lengths:
            cands_j.update(node_incident.get(nd, set()))
        total = 0.0
        for j in cands_j:
            du = lengths.get(int(eu[j]))
            dv = lengths.get(int(ev[j]))
            if du is None and dv is None:
                continue
            d_ij = min(x for x in (du, dv) if x is not None)
            if d_ij <= bandwidth:
                total += dc_arr[j] * (1.0 - d_ij / bandwidth) ** 2
        raw_scores[i] = total

        if (i + 1) % 2000 == 0 or (i + 1) == n_edges:
            print(f"  {i+1:,} / {n_edges:,} ({100*(i+1)/n_edges:.0f}%) — {time.perf_counter()-t0:.0f}s")

    log_scores = np.log1p(raw_scores)
    vmin, vmax = log_scores.min(), log_scores.max()
    nkde_score = (log_scores - vmin) / (vmax - vmin) if vmax > vmin else np.zeros(n_edges)

    edges_out["nkde_score"] = nkde_score
    print(f"NKDE done. Score range: [{nkde_score.min():.4f}, {nkde_score.max():.4f}]")
    return edges_out


def generate_candidates(edges: gpd.GeoDataFrame, config: dict) -> gpd.GeoDataFrame:
    """
    Interpolate candidate points every candidate_spacing_m along edges,
    excluding excluded_highways. No constraint filtering (base pipeline).
    Each candidate inherits nkde_score from its parent edge.
    Same logic as 03_candidates.ipynb before constraint filtering.
    """
    spacing = config["candidate_spacing_m"]
    excluded = set(config["excluded_highways"])

    def _is_excluded(hw):
        if hw is None or (isinstance(hw, float) and np.isnan(hw)):
            return False
        vals = hw if isinstance(hw, (list, tuple)) else [hw]
        return bool(excluded.intersection(str(h) for h in vals))

    kept = edges[~edges["highway"].apply(_is_excluded)].copy()
    print(f"Edges after highway filter: {len(kept):,} / {len(edges):,}")

    records = []
    for row in kept.itertuples(index=False):
        geom = row.geometry
        length = geom.length
        if length <= 0:
            continue
        for d in np.arange(0, length, spacing):
            records.append({
                "geometry": geom.interpolate(float(d)),
                "nkde_score": row.nkde_score,
            })

    cands = gpd.GeoDataFrame(records, geometry="geometry", crs=kept.crs)
    print(f"Candidates generated: {len(cands):,}")
    return cands


def build_coverage(candidates: gpd.GeoDataFrame, nodes: gpd.GeoDataFrame,
                   edges: gpd.GeoDataFrame, config: dict) -> dict:
    """
    Build a pandana Network and compute coverage dict {i: [j, ...]}
    where j covers demand point i within coverage_radius_m network distance.

    Uses the int32 node-id remapping fix from 04_optimization.ipynb
    (pandana requires int32 node ids; osmid values are too large on Windows).
    """
    radius = config["coverage_radius_m"]
    prefilter_r = radius + 200

    # ── Remap osmid → sequential int32 (pandana requirement) ───────────────
    nodes_w = nodes.copy()
    nodes_w["node_id"] = np.arange(len(nodes_w), dtype=np.int32)
    osmid_to_nodeid = dict(zip(nodes_w["osmid"], nodes_w["node_id"]))
    nodes_w = nodes_w.set_index("node_id")
    nodes_w["x"] = nodes_w.geometry.x
    nodes_w["y"] = nodes_w.geometry.y

    edges_w = edges.copy()
    if "length_m" not in edges_w.columns:
        edges_w["length_m"] = edges_w.geometry.length
    edges_w["from_id"] = edges_w["u"].map(osmid_to_nodeid)
    edges_w["to_id"]   = edges_w["v"].map(osmid_to_nodeid)
    edges_w = edges_w.dropna(subset=["from_id", "to_id"])
    edges_w["from_id"] = edges_w["from_id"].astype(np.int32)
    edges_w["to_id"]   = edges_w["to_id"].astype(np.int32)

    network = pandana.Network(
        node_x=nodes_w["x"],
        node_y=nodes_w["y"],
        edge_from=edges_w["from_id"],
        edge_to=edges_w["to_id"],
        edge_weights=edges_w[["length_m"]],
    )
    print(f"Pandana network: {len(nodes_w):,} nodes, {len(edges_w):,} edges")

    # ── nearest_pois over candidates ────────────────────────────────────────
    n_pois = min(200, len(candidates))
    x_s = pd.Series(candidates.geometry.x.values)
    y_s = pd.Series(candidates.geometry.y.values)

    network.set_pois("cand", maxdist=prefilter_r, maxitems=n_pois, x_col=x_s, y_col=y_s)
    combined = network.nearest_pois(radius, "cand", num_pois=n_pois, include_poi_ids=True)

    dist_cols = list(range(1, n_pois + 1))
    id_cols   = [f"poi{k}" for k in range(1, n_pois + 1)]

    raw_nodes = network.get_node_ids(candidates.geometry.x, candidates.geometry.y)
    node_arr = raw_nodes.reindex(candidates.index, fill_value=-1).astype(int).values

    coverage = {}
    for i in range(len(candidates)):
        nd = node_arr[i]
        if nd < 0 or nd not in combined.index:
            coverage[i] = [i]
            continue
        row = combined.loc[nd]
        dists = row[dist_cols]
        ids   = row[id_cols]
        valid = dists.notna() & (dists <= radius)
        js = ids[valid.values].dropna().astype(int).tolist()
        coverage[i] = js if js else [i]

    sizes = [len(v) for v in coverage.values()]
    print(f"Coverage: mean={np.mean(sizes):.1f}, max={max(sizes)}, min={min(sizes)} sites per demand point")
    return coverage


def run_mclp(candidates: gpd.GeoDataFrame, coverage: dict, p: int):
    """
    Maximize sum(nkde_score_i * y_i) subject to:
      sum(x_j for j in coverage[i]) >= y_i  for all i
      sum(x_j) == p
      x_j in {0,1},  y_i in [0,1]
    """
    t0 = time.time()
    n = len(candidates)
    nkde = candidates["nkde_score"].values

    prob = pulp.LpProblem(f"MCLP_p{p}", pulp.LpMaximize)
    x = pulp.LpVariable.dicts("x", range(n), cat="Binary")
    y = pulp.LpVariable.dicts("y", range(n), lowBound=0, upBound=1, cat="Continuous")

    prob += pulp.lpSum(nkde[i] * y[i] for i in range(n))

    for i, js in coverage.items():
        prob += pulp.lpSum(x[j] for j in js) >= y[i]

    prob += pulp.lpSum(x[j] for j in range(n)) == p

    prob.solve(pulp.PULP_CBC_CMD(msg=0, gapRel=0.01, timeLimit=300))

    sel_idx = [j for j in range(n) if (pulp.value(x[j]) or 0) > 0.5]
    total_dem = nkde.sum()
    cov_dem = sum(nkde[i] for i in range(n) if (pulp.value(y[i]) or 0) > 0.5)
    pct = cov_dem / total_dem * 100 if total_dem > 0 else 0.0

    sel_gdf = candidates.iloc[sel_idx].copy().reset_index(drop=True)
    print(f"  p={p}: {len(sel_gdf)} sites | {pct:.1f}% demand covered | {time.time()-t0:.1f}s")
    return sel_gdf, pct


def run_pipeline(config: dict) -> dict:
    """Run all pipeline steps in order and save outputs."""
    prefix   = config["output_prefix"]
    proc_dir = PROJECT_ROOT / "data/processed"
    out_dir  = PROJECT_ROOT / "data/outputs"
    proc_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    demand, target_crs = load_demand(config)
    G, nodes, edges = pull_network(config, target_crs)
    edges_nkde = compute_nkde(edges, demand, config, G)
    candidates = generate_candidates(edges_nkde, config)
    coverage = build_coverage(candidates, nodes, edges_nkde, config)

    results = {}
    for p in config["p_values"]:
        sel_gdf, pct = run_mclp(candidates, coverage, p)
        results[p] = {"selected_gdf": sel_gdf, "pct_covered": pct}
        out_gpkg = proc_dir / f"{prefix}_mclp_p{p}.gpkg"
        sel_gdf.to_crs("EPSG:4326").to_file(out_gpkg, driver="GPKG")
        print(f"    Saved → {out_gpkg.name}")

    # ── Coverage curve ──────────────────────────────────────────────────────
    p_vals = sorted(results.keys())
    pcts   = [results[p]["pct_covered"] for p in p_vals]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(p_vals, pcts, "o-", color="steelblue", linewidth=2, markersize=8)
    ax.set_xlabel("Number of PUDO Sites (p)", fontsize=12)
    ax.set_ylabel("% Weighted Demand Covered", fontsize=12)
    ax.set_title(f"MCLP Coverage Curve — {config['place_name']}", fontsize=13)
    ax.set_xticks(p_vals)
    ax.grid(True, alpha=0.3)
    for p, pct in zip(p_vals, pcts):
        ax.annotate(f"{pct:.1f}%", (p, pct), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=10)
    plt.tight_layout()
    curve_path = out_dir / f"{prefix}_coverage_curve.png"
    plt.savefig(curve_path, dpi=150)
    plt.close()
    print(f"Coverage curve → {curve_path.name}")

    # ── Map of p=100 selected sites ─────────────────────────────────────────
    p_map = 100 if 100 in results else p_vals[-1]
    sel_p = results[p_map]["selected_gdf"]

    fig, ax = plt.subplots(figsize=(12, 12))
    edges_nkde.plot(ax=ax, color="#cccccc", linewidth=0.4, zorder=1)
    sel_p.plot(ax=ax, color="crimson", markersize=8, zorder=2, label=f"p={p_map} sites")
    ax.set_title(f"MCLP Selected Sites (p={p_map}) — {config['place_name']}", fontsize=14)
    ax.set_axis_off()
    ax.legend(fontsize=11)
    plt.tight_layout()
    map_path = out_dir / f"{prefix}_sites_p{p_map}.png"
    plt.savefig(map_path, dpi=150)
    plt.close()
    print(f"Sites map → {map_path.name}")

    return results


In [4]:
results = run_pipeline(CONFIG)

Demand rows loaded: 152,547
Detected UTM CRS: EPSG:32629  (median lon=-8.6132, lat=41.1543)
Network: 5,038 nodes, 10,540 edges
Snapping 152,547 demand points to edges …
  50,000 / 152,547 (33%)
  100,000 / 152,547 (66%)
  150,000 / 152,547 (98%)
  152,547 / 152,547 (100%)
Computing NKDE over 10,540 edges (bandwidth=500 m) …
  2,000 / 10,540 (19%) — 1s
  4,000 / 10,540 (38%) — 1s
  6,000 / 10,540 (57%) — 2s
  8,000 / 10,540 (76%) — 3s
  10,000 / 10,540 (95%) — 4s
  10,540 / 10,540 (100%) — 4s
NKDE done. Score range: [0.0000, 1.0000]
Edges after highway filter: 10,298 / 10,540
Candidates generated: 10,307
Pandana network: 5,038 nodes, 10,540 edges
Coverage: mean=120.3, max=200, min=3 sites per demand point
  p=20: 20 sites | 43.0% demand covered | 26.3s
    Saved → porto_mclp_p20.gpkg
  p=40: 40 sites | 68.7% demand covered | 35.7s
    Saved → porto_mclp_p40.gpkg
  p=55: 55 sites | 81.6% demand covered | 53.0s
    Saved → porto_mclp_p55.gpkg
  p=70: 70 sites | 90.7% demand covered | 33.9